# Ensemble Methods — Random Forest & Gradient Boosting

**Course**: CMOR 438 / INDE 577 — Data Science & Machine Learning  
**Dataset**: International Football Results (1872–present)  
**Author**: Neriah29  

---

## Why Ensembles?

A single Decision Tree is powerful but unstable — small changes in the training data produce very different trees, and deep trees overfit badly. The fix is simple in concept:

> *Train many trees and combine their answers. Individual errors cancel out.*

This notebook covers two fundamentally different ways to do this.

---

## Random Forest — Bagging

Train many trees **independently and in parallel**, each on a random subset of data and features. Prediction = average probability across all trees.

Two sources of deliberate randomness:
- **Bootstrap sampling**: each tree trains on a random sample drawn *with replacement* — so some matches appear multiple times, others not at all
- **Feature subsampling**: at each split, only √(n_features) randomly chosen features are considered — trees can't all ask the same questions

Result: 100 diverse trees whose errors are uncorrelated. When you average them, the noise cancels and accuracy improves dramatically. This technique is called **Bagging** (Bootstrap Aggregating).

---

## Gradient Boosting — Sequential Correction

Train trees **sequentially**, each one correcting the mistakes of all previous trees.

At each step:
1. Compute **pseudo-residuals** = how wrong the current ensemble is for each sample (y − p)
2. Fit a shallow tree to predict those residuals
3. Add that tree's predictions to the running score, scaled by the learning rate

This is **gradient descent in function space** — instead of adjusting weights, we're adding trees. Each new tree is a step in the direction that reduces the loss most.

---

## Key Comparison

| | Random Forest | Gradient Boosting |
|---|---|---|
| Trees trained | In parallel | Sequentially |
| Strategy | Reduce variance | Reduce bias |
| Speed | Fast | Slower |
| Overfitting risk | Low | Higher — needs tuning |
| Typical accuracy | Very good | Slightly better with tuning |


---
## Imports

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.model_selection import train_test_split
from sklearn.metrics import confusion_matrix, classification_report, roc_curve, auc

import sys
sys.path.insert(0, '../../src')
from football_ml.supervised_learning.ensemble import (
    RandomForestClassifier,
    GradientBoostingClassifier,
)
from football_ml.supervised_learning.decision_tree import DecisionTreeClassifier
from football_ml.supervised_learning.logistic_regression import LogisticRegression

sns.set_theme(style='whitegrid', palette='muted')
plt.rcParams['figure.dpi'] = 120
SEED = 42

---
## 1. Load & Engineer Features

In [ ]:
df = pd.read_csv('../../data/results.csv')
df['date'] = pd.to_datetime(df['date'])
df = df.sort_values('date').reset_index(drop=True)
df['home_win'] = (df['home_score'] > df['away_score']).astype(int)

def compute_team_rolling_stats(df, window=10):
    home_log = df[['date', 'home_team', 'home_score', 'away_score']].copy()
    home_log.columns = ['date', 'team', 'scored', 'conceded']
    away_log = df[['date', 'away_team', 'away_score', 'home_score']].copy()
    away_log.columns = ['date', 'team', 'scored', 'conceded']
    team_log = pd.concat([home_log, away_log]).sort_values('date').reset_index(drop=True)
    team_log['rolling_scored'] = (
        team_log.groupby('team')['scored']
        .transform(lambda x: x.shift(1).rolling(window, min_periods=1).mean())
    )
    team_log['rolling_conceded'] = (
        team_log.groupby('team')['conceded']
        .transform(lambda x: x.shift(1).rolling(window, min_periods=1).mean())
    )
    return team_log.drop_duplicates(subset=['date', 'team'], keep='last').set_index(['date', 'team'])

team_stats = compute_team_rolling_stats(df)

def get_stat(row, team_col, stat_col):
    try:
        return team_stats.loc[(row['date'], row[team_col]), stat_col]
    except KeyError:
        return np.nan

df['home_goals_rolling']    = df.apply(lambda r: get_stat(r, 'home_team', 'rolling_scored'), axis=1)
df['home_conceded_rolling'] = df.apply(lambda r: get_stat(r, 'home_team', 'rolling_conceded'), axis=1)
df['away_goals_rolling']    = df.apply(lambda r: get_stat(r, 'away_team', 'rolling_scored'), axis=1)
df['away_conceded_rolling'] = df.apply(lambda r: get_stat(r, 'away_team', 'rolling_conceded'), axis=1)

home_wins = df.groupby('home_team').apply(lambda g: (g['home_score'] > g['away_score']).mean()).rename('home_win_rate')
away_wins = df.groupby('away_team').apply(lambda g: (g['away_score'] > g['home_score']).mean()).rename('away_win_rate')
df = df.join(home_wins, on='home_team').join(away_wins, on='away_team')
df['neutral'] = df['neutral'].astype(int)

feature_cols = [
    'home_goals_rolling', 'away_goals_rolling',
    'home_conceded_rolling', 'away_conceded_rolling',
    'home_win_rate', 'away_win_rate', 'neutral'
]
df_clean = df[feature_cols + ['home_win']].dropna()
print(f'Dataset: {df_clean.shape[0]} matches | Home win rate: {df_clean["home_win"].mean():.1%}')

---
## 2. Prepare Data

In [ ]:
X = df_clean[feature_cols].values
y = df_clean['home_win'].values

# No scaling needed — tree-based methods are scale-invariant
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=SEED, stratify=y
)
print(f'Train: {X_train.shape} | Test: {X_test.shape}')

---
## 3. Train Both Models

In [ ]:
rf = RandomForestClassifier(
    n_estimators=100, max_depth=8,
    max_features='sqrt', random_state=SEED
).fit(X_train, y_train)

gb = GradientBoostingClassifier(
    n_estimators=100, learning_rate=0.1,
    max_depth=3, random_state=SEED
).fit(X_train, y_train)

print('Random Forest:')
print(f'  Train accuracy: {rf.score(X_train, y_train):.3f}')
print(f'  Test  accuracy: {rf.score(X_test,  y_test):.3f}')
print()
print('Gradient Boosting:')
print(f'  Train accuracy: {gb.score(X_train, y_train):.3f}')
print(f'  Test  accuracy: {gb.score(X_test,  y_test):.3f}')

---
## 4. Gradient Boosting Loss Curve

In [ ]:
fig, ax = plt.subplots(figsize=(8, 4))
ax.plot(gb.loss_history_, color='#E87272', linewidth=2)
ax.fill_between(range(len(gb.loss_history_)), gb.loss_history_, alpha=0.1, color='#E87272')
ax.axhline(np.log(2), color='gray', linestyle='--', linewidth=1.2,
           label=f'Random baseline ({np.log(2):.3f})')
ax.set_xlabel('Boosting Round (trees added)', fontsize=12)
ax.set_ylabel('Log Loss', fontsize=12)
ax.set_title('Gradient Boosting — Loss per Round', fontsize=13, fontweight='bold')
ax.legend(fontsize=10)
plt.tight_layout()
plt.show()

Each point on this curve represents one tree being added. The loss drops smoothly — each tree corrects the residuals of everything before it. Compare this to the MLP loss curve: the shape is similar because both are gradient descent, just in different spaces.

---
## 5. Feature Importances — Both Models

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(13, 4))

for ax, model, title, color in zip(
    axes,
    [rf, gb],
    ['Random Forest — Feature Importances', 'Gradient Boosting — Feature Importances'],
    ['#4878CF', '#E87272']
):
    importances = model.feature_importances_
    sorted_idx  = np.argsort(importances)[::-1]
    ax.barh(
        [feature_cols[i] for i in sorted_idx],
        importances[sorted_idx],
        color=color, edgecolor='white'
    )
    ax.set_xlabel('Importance', fontsize=11)
    ax.set_title(title, fontsize=12, fontweight='bold')
    ax.invert_yaxis()

plt.tight_layout()
plt.show()

---
## 6. Effect of n_estimators

In [ ]:
estimator_counts = [1, 5, 10, 20, 50, 100]
rf_scores, gb_scores = [], []

for n in estimator_counts:
    rf_n = RandomForestClassifier(n_estimators=n, max_depth=8, random_state=SEED).fit(X_train, y_train)
    gb_n = GradientBoostingClassifier(n_estimators=n, learning_rate=0.1, random_state=SEED).fit(X_train, y_train)
    rf_scores.append(rf_n.score(X_test, y_test))
    gb_scores.append(gb_n.score(X_test, y_test))

fig, ax = plt.subplots(figsize=(8, 4))
ax.plot(estimator_counts, rf_scores, label='Random Forest', color='#4878CF', linewidth=2, marker='o')
ax.plot(estimator_counts, gb_scores, label='Gradient Boosting', color='#E87272', linewidth=2, marker='o')
ax.set_xlabel('Number of Trees', fontsize=12)
ax.set_ylabel('Test Accuracy', fontsize=12)
ax.set_title('Accuracy vs Number of Trees', fontsize=13, fontweight='bold')
ax.legend(fontsize=10)
plt.tight_layout()
plt.show()

---
## 7. Full Model Comparison — ROC Curves

In [ ]:
from sklearn.preprocessing import StandardScaler
from football_ml.supervised_learning.mlp import MLP

scaler = StandardScaler()
X_train_sc = scaler.fit_transform(X_train)
X_test_sc  = scaler.transform(X_test)

models = {
    'Logistic Regression': (LogisticRegression(learning_rate=0.1, n_epochs=1000, random_state=SEED), X_train_sc, X_test_sc),
    'Decision Tree':       (DecisionTreeClassifier(max_depth=5), X_train, X_test),
    'Random Forest':       (rf, X_train, X_test),
    'Gradient Boosting':   (gb, X_train, X_test),
    'MLP':                 (MLP(hidden_layer_sizes=(64,32), learning_rate=0.01, n_epochs=500, random_state=SEED), X_train_sc, X_test_sc),
}

fig, ax = plt.subplots(figsize=(8, 6))
colors = ['#95a5a6', '#2ecc71', '#4878CF', '#E87272', '#9b59b6']

for (name, (model, Xtr, Xte)), color in zip(models.items(), colors):
    model.fit(Xtr, y_train)
    probs = model.predict_proba(Xte)
    fpr, tpr, _ = roc_curve(y_test, probs)
    roc_auc = auc(fpr, tpr)
    ax.plot(fpr, tpr, color=color, linewidth=2, label=f'{name} (AUC={roc_auc:.3f})')

ax.plot([0,1],[0,1],'gray',linestyle='--',linewidth=1,label='Random')
ax.set_xlabel('False Positive Rate', fontsize=12)
ax.set_ylabel('True Positive Rate', fontsize=12)
ax.set_title('ROC Curve — All Models', fontsize=13, fontweight='bold')
ax.legend(fontsize=9, loc='lower right')
plt.tight_layout()
plt.show()

---
## 8. Discussion

### Random Forest
- **Strengths**: robust, hard to overfit, parallel training, great feature importances, no scaling needed
- **Weaknesses**: less interpretable than a single tree, slower prediction than a single model
- **Football**: averaging 100 trees smooths out the noise in individual matches — more reliable than any single tree

### Gradient Boosting
- **Strengths**: often the most accurate tree-based method, explicitly optimizes the loss function
- **Weaknesses**: sequential training is slower, more sensitive to hyperparameters, can overfit if learning rate is too high or trees too deep
- **Football**: the sequential correction approach is powerful but requires careful tuning

### Why ensembles work

The fundamental reason: **uncorrelated errors cancel**. If 100 different models each make random errors in different directions, their average is much closer to truth than any individual. The deliberate randomness in Random Forest (bootstrap + feature subsampling) ensures trees are diverse enough that their errors don't all point the same way.

---
## Summary

| | Random Forest | Gradient Boosting |
|---|---|---|
| **Strategy** | Bagging — parallel independent trees | Boosting — sequential correction |
| **Randomness** | Bootstrap + feature subsampling | None — deterministic given seed |
| **Loss function** | Implicit (Gini per tree) | Explicit (log-loss, gradient descent) |
| **Key hyperparameters** | n_estimators, max_depth, max_features | n_estimators, learning_rate, max_depth |
| **Needs scaling** | No | No |
| **Football suitability** | Good | Good — slightly better with tuning |
